In [7]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from dotenv import load_dotenv
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

pd.set_option("display.max_columns", None)

In [8]:
load_dotenv("../../.env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

print("DB_USER:", DB_USER)
print("DB_HOST:", DB_HOST)
print("DB_PORT:", DB_PORT)
print("DB_NAME:", DB_NAME)

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(connection_url)

DB_USER: postgres
DB_HOST: localhost
DB_PORT: 5432
DB_NAME: churn_project


In [11]:
query = """
SELECT *
FROM analytics_gold.gold_churn_model_input
"""

df = pd.read_sql(query, engine)

df.head()

,customer_id,churn,tenure,city_tier,warehouse_to_home,hour_spend_on_app,number_of_device_registered,satisfaction_score,number_of_address,complain,order_amount_hike_from_last_year,coupon_used,order_count,day_since_last_order,cashback_amount,preferred_login_device,preferred_payment_mode,gender,preferred_order_cat,marital_status,tenure_missing_flag,warehouse_to_home_missing_flag,hour_spend_on_app_missing_flag,day_since_last_order_missing_flag,is_new_customer,low_satisfaction_flag,has_complaint,inactive_customer_flag,high_cashback_customer_flag,batch_id,loaded_at,loaded_by
0,50001,1,4.0,3,6.0,3.0,3.0,2,9.0,1,11.0,1.0,1.0,5.0,160.0,mobile phone,debit card,female,laptop & accessory,single,0,0,0,0,1,1,1,0,0,f190cdea-1610-40b4-8591-813dff94068a,2026-06-17 14:50:23.631095,local_user
1,50002,1,0.0,1,8.0,3.0,4.0,3,7.0,1,15.0,0.0,1.0,0.0,121.0,phone,upi,male,mobile,single,1,0,0,0,1,0,1,0,0,f190cdea-1610-40b4-8591-813dff94068a,2026-06-17 14:50:23.631095,local_user
2,50003,1,0.0,1,30.0,2.0,4.0,3,6.0,1,14.0,0.0,1.0,3.0,120.0,phone,debit card,male,mobile,single,1,0,0,0,1,0,1,0,0,f190cdea-1610-40b4-8591-813dff94068a,2026-06-17 14:50:23.631095,local_user
3,50004,1,0.0,3,15.0,2.0,4.0,5,8.0,0,23.0,0.0,1.0,3.0,134.0,phone,debit card,male,laptop & accessory,single,0,0,0,0,1,0,0,0,0,f190cdea-1610-40b4-8591-813dff94068a,2026-06-17 14:50:23.631095,local_user
4,50005,1,0.0,1,12.0,0.0,3.0,5,3.0,0,11.0,1.0,1.0,3.0,130.0,phone,cc,male,mobile,single,0,0,1,0,1,0,0,0,0,f190cdea-1610-40b4-8591-813dff94068a,2026-06-17 14:50:23.631095,local_user


In [12]:
df_ml = df.copy()

In [13]:
df_ml["preferred_payment_mode"] = df_ml["preferred_payment_mode"].replace({
    "cod": "cash on delivery",
    "cc": "credit card"
})

df_ml["preferred_login_device"] = df_ml["preferred_login_device"].replace({
    "phone": "mobile phone"
})

In [14]:
for col in ["preferred_payment_mode", "preferred_login_device"]:
    print(f"\n{col}")
    print(df_ml[col].value_counts())


preferred_payment_mode
preferred_payment_mode
debit card          2314
credit card         1774
e wallet             614
cash on delivery     514
upi                  414
Name: count, dtype: int64

preferred_login_device
preferred_login_device
mobile phone    3996
computer        1634
Name: count, dtype: int64


In [15]:
drop_cols = [
    "customer_id",
    "batch_id",
    "loaded_at",
    "loaded_by"
]

df_ml = df_ml.drop(columns=drop_cols)

In [16]:
X = df_ml.drop(columns=["churn"])
y = df_ml["churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5630, 27)
y shape: (5630,)


In [17]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumeric features:")
print(numeric_features)

Categorical features:
['preferred_login_device', 'preferred_payment_mode', 'gender', 'preferred_order_cat', 'marital_status']

Numeric features:
['tenure', 'city_tier', 'warehouse_to_home', 'hour_spend_on_app', 'number_of_device_registered', 'satisfaction_score', 'number_of_address', 'complain', 'order_amount_hike_from_last_year', 'coupon_used', 'order_count', 'day_since_last_order', 'cashback_amount', 'tenure_missing_flag', 'warehouse_to_home_missing_flag', 'hour_spend_on_app_missing_flag', 'day_since_last_order_missing_flag', 'is_new_customer', 'low_satisfaction_flag', 'has_complaint', 'inactive_customer_flag', 'high_cashback_customer_flag']


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\ny_test distribution:")
print(y_test.value_counts(normalize=True) * 100)

X_train shape: (4504, 27)
X_test shape: (1126, 27)

y_train distribution:
churn
0    83.170515
1    16.829485
Name: proportion, dtype: float64

y_test distribution:
churn
0    83.12611
1    16.87389
Name: proportion, dtype: float64


In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

## Preprocessing Summary

The Gold table was loaded from PostgreSQL and prepared for machine learning.

Several categorical values were standardized before modeling. For example, `cod` was merged with `cash on delivery`, `cc` was merged with `credit card`, and `phone` was merged with `mobile phone`.

Metadata columns such as `customer_id`, `batch_id`, `loaded_at`, and `loaded_by` were removed because they are not behavioral predictors.

The target variable is `churn`, while the remaining columns are used as model features. The dataset was split into training and test sets using an 80/20 split. Stratified sampling was applied to preserve the churn distribution in both sets.

A preprocessing pipeline was defined using `StandardScaler` for numeric features and `OneHotEncoder` for categorical features.